<a href="https://colab.research.google.com/github/aritazaman/clearinghouse/blob/main/bert_masked_language_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
#testing t4 gpu
import torch
print(torch.cuda.is_available())

True


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cases.json to cases.json


In [ ]:
#checking to see if cases.json is in the directory
import os
print(os.listdir())

['.config', 'cases.json', 'sample_data']


In [ ]:
with open("cases.json", "r") as f:
    cases = json.load(f)
# cases = cases[:1000]

print(f"Loaded {len(cases)} cases")

Loaded 13399 cases


In [ ]:
texts = []
for case in cases:
    text = case.get("summary") or case.get("name")  #fallback to case name if summary is empty
    if text:  #skip empty
        texts.append({"text": text})
print(f"Total texts: {len(texts)}")


Total texts: 13364


In [ ]:
#create HF dataset
dataset = Dataset.from_list(texts)

#load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/13364 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#data collator for MLM (handles random masking)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
    #0.15 is default, but is this the ideal probability?
)


In [ ]:
#training arguments
training_args = TrainingArguments(
    output_dir="./results",
    #whats the ideal number of epochs?
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
#trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


In [ ]:
trainer.train()

Step,Training Loss
100,1.633086
200,1.457959
300,1.383392
400,1.311939
500,1.285288
600,1.265666
700,1.270629
800,1.237902
900,1.264492
1000,1.198901


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5013, training_loss=1.0819562259082807, metrics={'train_runtime': 1420.2235, 'train_samples_per_second': 28.229, 'train_steps_per_second': 3.53, 'total_flos': 2638101838694400.0, 'train_loss': 1.0819562259082807, 'epoch': 3.0})

In [ ]:
!zip -r trained_bert.zip ./results
from google.colab import files
files.download("trained_bert.zip")

  adding: results/ (stored 0%)
  adding: results/checkpoint-5013/ (stored 0%)
  adding: results/checkpoint-5013/trainer_state.json (deflated 73%)
  adding: results/checkpoint-5013/scheduler.pt (deflated 62%)
  adding: results/checkpoint-5013/model.safetensors (deflated 7%)
  adding: results/checkpoint-5013/optimizer.pt (deflated 8%)
  adding: results/checkpoint-5013/tokenizer.json (deflated 71%)
  adding: results/checkpoint-5013/rng_state.pth (deflated 26%)
  adding: results/checkpoint-5013/config.json (deflated 52%)
  adding: results/checkpoint-5013/training_args.bin (deflated 53%)
  adding: results/checkpoint-5013/tokenizer_config.json (deflated 42%)
  adding: results/checkpoint-5000/ (stored 0%)
  adding: results/checkpoint-5000/trainer_state.json (deflated 73%)
  adding: results/checkpoint-5000/scheduler.pt (deflated 61%)
  adding: results/checkpoint-5000/model.safetensors (deflated 7%)
  adding: results/checkpoint-5000/optimizer.pt (deflated 8%)
  adding: results/checkpoint-5000/t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

#saving the trained model and tokenizer in a folder
model.save_pretrained("./bert_finetuned")
tokenizer.save_pretrained("./bert_finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_finetuned/tokenizer_config.json', './bert_finetuned/tokenizer.json')

In [ ]:
#test
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="./bert_finetuned",
    tokenizer="./bert_finetuned"
)

print(fill_mask("Plaintiffs in Penguin Random House v. Robbins argued that the Iowa law violated the First Amendment and the [MASK] Amendment."))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[{'score': 0.7961530685424805, 'token': 15276, 'token_str': 'fourteenth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fourteenth amendment.'}, {'score': 0.05883459001779556, 'token': 3587, 'token_str': 'fifth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fifth amendment.'}, {'score': 0.05036964640021324, 'token': 5964, 'token_str': 'eighth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the eighth amendment.'}, {'score': 0.044632438570261, 'token': 2959, 'token_str': 'fourth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fourth amendment.'}, {'score': 0.011102002114057541, 'token': 6400, 'token_str': '14th', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law

[{'score': 0.7961530685424805, 'token': 15276, 'token_str': 'fourteenth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fourteenth amendment.'},

{'score': 0.05883459001779556, 'token': 3587, 'token_str': 'fifth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fifth amendment.'},

{'score': 0.05036964640021324, 'token': 5964, 'token_str': 'eighth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the eighth amendment.'},

{'score': 0.044632438570261, 'token': 2959, 'token_str': 'fourth', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the fourth amendment.'},

{'score': 0.011102002114057541, 'token': 6400, 'token_str': '14th', 'sequence': 'plaintiffs in penguin random house v. robbins argued that the iowa law violated the first amendment and the 14th amendment.'}]